In [1]:
from __future__ import annotations

import argparse
import importlib.util
import logging
import sys
import time
import types
from pathlib import Path

from src.utils import pmf_utils
import pandas as pd
import matplotlib.pyplot as plt
import importlib, src.ddm.ddm
importlib.reload(src.ddm.ddm)

import numpy as np
import torch

from config import dir_config

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)


In [2]:
processed_dir = Path(dir_config.data.processed)
ddm_dir = processed_dir / 'ddm'

session_metadata = pd.read_csv(processed_dir / "sessions_metadata.csv")
behavior_df = pd.read_csv(ddm_dir / "behavior_data.csv")

In [3]:
def plot_ddm_fit(data, sim, coherences=None, title="DDM Fit"):
    """
    Overlay observed data (dots) and model predictions (lines).

    data : DataFrame or dict with keys 'rt', 'choice', 'coherence'
    sim  : DataFrame or dict with keys 'rt', 'choice', 'coherence'
    """
    if not isinstance(data, pd.DataFrame):
        data = pd.DataFrame(data)
    if not isinstance(sim, pd.DataFrame):
        sim = pd.DataFrame(sim)

    data = data.copy()
    sim  = sim.copy()
    data["coherence"] = data["coherence"].round(2)
    sim["coherence"]  = sim["coherence"].round(2)

    cohs = sorted(coherences if coherences is not None else data["coherence"].unique())

    def summarize(df):
        p_upper, mean_rt_upper, mean_rt_lower = [], [], []
        for c in cohs:
            subset = df[df["coherence"] == c].dropna(subset=["rt", "choice"])
            p_upper.append((subset["choice"] == 1).mean() if len(subset) > 0 else np.nan)
            upper = subset[subset["choice"] == 1]["rt"]
            lower = subset[subset["choice"] == 0]["rt"]
            mean_rt_upper.append(upper.mean() if len(upper) > 0 else np.nan)
            mean_rt_lower.append(lower.mean() if len(lower) > 0 else np.nan)
        return np.array(p_upper), np.array(mean_rt_upper), np.array(mean_rt_lower)

    d_p, d_rt1, d_rt0 = summarize(data)
    s_p, s_rt1, s_rt0 = summarize(sim)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
    fig.suptitle(title)

    # --- Psychometric ---
    ax1.plot(cohs, s_p, color="steelblue", label="model")
    ax1.scatter(cohs, d_p, color="steelblue", zorder=5, label="data")
    ax1.axhline(0.5, color="gray", linestyle="--", linewidth=0.8)
    ax1.axvline(0.0, color="gray", linestyle="--", linewidth=0.8)
    ax1.set(xlabel="Coherence", ylabel="P(upper boundary)", title="Psychometric", ylim=(0, 1))
    ax1.legend()

    # --- Chronometric ---
    ax2.plot(cohs, s_rt1, color="steelblue", label="model upper")
    ax2.plot(cohs, s_rt0, color="tomato",    label="model lower")
    ax2.scatter(cohs, d_rt1, color="steelblue", zorder=5, label="data upper")
    ax2.scatter(cohs, d_rt0, color="tomato",    zorder=5, label="data lower")
    ax2.axvline(0.0, color="gray", linestyle="--", linewidth=0.8)
    ax2.set(xlabel="Coherence", ylabel="Mean RT (s)", title="Chronometric")
    ax2.legend()

    plt.tight_layout()
    # return fig

In [4]:
def load_behavior_data(ddm_dir: Path) -> pd.DataFrame:

    behavior_path = ddm_dir / "behavior_data.csv"

    if not behavior_path.exists():
        raise FileNotFoundError(f"Missing file: {behavior_path}")

    return pd.read_csv(behavior_path)

def build_grid(behavior_df: pd.DataFrame) -> list[dict]:

    session_ids = np.sort(behavior_df["session_id"].unique())
    prior_blocks = np.sort(behavior_df["prior_block"].unique())

    variants = [
        (False, False),
        (False, True),
        (True,  False),
        (True,  True),
    ]

    return [
        {
            "session_id": session_id,
            "prior_block": prior_block,
            "enable_leak": enable_leak,
            "enable_time_constant": enable_time_constant,
        }
        for enable_leak, enable_time_constant in variants
        for session_id in session_ids
        for prior_block in prior_blocks
    ]

def get_job(grid: list[dict], job_id: int) -> dict:

    if job_id >= len(grid):
        raise ValueError(
            f"job_id {job_id} out of bounds "
            f"(max={len(grid)-1})"
        )

    return grid[job_id]

In [6]:
session_ids = session_metadata["session_id"].tolist()
grid = build_grid(behavior_df)
job_lookup = {job_id: get_job(grid, job_id) for job_id in range(len(grid))}
stem_to_job_id = {
    (
        job["session_id"],
        job["prior_block"],
        job["enable_leak"],
        job["enable_time_constant"],
    ): job_id
    for job_id, job in job_lookup.items()
}

missing_job_ids = set()
for sub_dir in ddm_dir.iterdir():
    if not sub_dir.is_dir():
        continue

    # ----------------------------
    # parse variant from folder name
    # ----------------------------
    leak = "leak-1" in sub_dir.name
    tc = "tc-1" in sub_dir.name

    expected = {
        (sid, block, leak, tc)
        for sid in session_ids
        for block in [0, 1]
    }

    found = set()

    for model in sub_dir.rglob("*.pkl"):

        try:
            session = model.stem.split("_prior_block_")[0]
            block = int(model.stem.split("_prior_block_")[1])

            found.add((session, block, leak, tc))

        except Exception as e:
            print(f"[PARSE ERROR] {model}: {e}")

    missing = expected - found
    extra = found - expected

    print(
        f"\n=== {sub_dir.name} ===\n"
        f"Total files: {len(list(sub_dir.rglob('*.pkl')))}\n"
        f"Expected: {len(expected)}\n"
        f"Missing: {len(missing)}\n"
        f"Extra: {len(extra)}"
    )

    # ----------------------------
    # missing reporting
    # ----------------------------
    if missing:
        print("\nMISSING DETAILS:")
        for m in sorted(missing):
            job_id = stem_to_job_id.get(m)

            print(f"{m} → job_id={job_id}")

            if job_id is not None:
                missing_job_ids.add(job_id)


print("\n=== SUMMARY ===")
print("Missing job IDs:", sorted(missing_job_ids))


=== leak-0_tc-1 ===
Total files: 90
Expected: 90
Missing: 0
Extra: 0

=== leak-1_tc-1 ===
Total files: 90
Expected: 90
Missing: 0
Extra: 0

=== leak-0_tc-0 ===
Total files: 90
Expected: 90
Missing: 0
Extra: 0

=== leak-1_tc-0 ===
Total files: 90
Expected: 90
Missing: 0
Extra: 0

=== SUMMARY ===
Missing job IDs: []


# Session Wise Model Fits